# Load User Metadata

In [2]:
import pandas as pd
import numpy as np

df_users_final = pd.read_parquet("./datasets/final_outputs/df_users_final.parquet")

df_users = df_users_final.copy()
print(df_users.shape)
print(df_users.columns.tolist())
df_users_final.head()


(20000, 26)
['id', 'label', 'split', 'participation_degree', 'created_at', 'description', 'location', 'name', 'pinned_tweet_id', 'profile_image_url', 'protected', 'url', 'username', 'verified', 'withheld', 'entities.url.urls', 'entities.description.urls', 'public_metrics.followers_count', 'public_metrics.following_count', 'public_metrics.tweet_count', 'public_metrics.listed_count', 'entities', 'entities.description.mentions', 'entities.description.hashtags', 'entities.description.cashtags', 'withheld.country_codes']


,id,label,split,participation_degree,created_at,description,location,name,pinned_tweet_id,profile_image_url,...,entities.description.urls,public_metrics.followers_count,public_metrics.following_count,public_metrics.tweet_count,public_metrics.listed_count,entities,entities.description.mentions,entities.description.hashtags,entities.description.cashtags,withheld.country_codes
0,u1217628182611927040,human,test,1893,2020-01-16 02:02:55+00:00,Theoretical Computer Scientist. See also https...,"Cambridge, MA",Boaz Barak,NaN,https://pbs.twimg.com/profile_images/125226236...,...,"[{'display_url': 'windowsontheory.org', 'end':...",7316,215,3098,69,NaN,None,None,None,None
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,,🇬🇧,Grian,1.143808e+18,https://pbs.twimg.com/profile_images/100773461...,...,None,238254,293,1400,448,NaN,None,None,None,None
2,u1341789703633178624,bot,test,159,2020-12-23 16:56:30+00:00,,NaN,mo,NaN,https://abs.twimg.com/sticky/default_profile_i...,...,None,0,136,6,0,NaN,None,None,None,None
3,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",NaN,AK,NaN,https://pbs.twimg.com/profile_images/145119163...,...,None,45541,1206,9194,605,NaN,None,None,None,None
4,u1099603759259238400,human,test,326,2019-02-24 09:35:40+00:00,,NaN,Ryuu09,NaN,https://pbs.twimg.com/profile_images/121992857...,...,None,8,289,0,0,NaN,None,None,None,None


# Drop duplicates 

In [3]:
# standardize user id
df_users["id"] = df_users["id"].astype(str).str.strip()

# remove duplicate users
df_users = df_users.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(df_users.shape)
print(df_users["id"].nunique())

(20000, 26)
20000


# Check for Sparsity and drop columns with over 90% missing data

In [4]:
# overall sparsity summary for all columns
sparsity_df = pd.DataFrame({
    "column": df_users.columns,
    "dtype": [str(df_users[c].dtype) for c in df_users.columns],
    "missing_count": [df_users[c].isna().sum() for c in df_users.columns],
    "missing_ratio": [df_users[c].isna().mean() for c in df_users.columns],
})

sparsity_df["zero_count"] = [
    (df_users[c] == 0).sum() if pd.api.types.is_numeric_dtype(df_users[c]) else None
    for c in df_users.columns
]

sparsity_df["zero_ratio"] = [
    (df_users[c] == 0).mean() if pd.api.types.is_numeric_dtype(df_users[c]) else None
    for c in df_users.columns
]

sparsity_df["empty_string_count"] = [
    (df_users[c].astype(str).str.strip() == "").sum() if df_users[c].dtype == "object" else None
    for c in df_users.columns
]

sparsity_df["empty_string_ratio"] = [
    (df_users[c].astype(str).str.strip() == "").mean() if df_users[c].dtype == "object" else None
    for c in df_users.columns
]

sparsity_df = sparsity_df.sort_values(
    by=["missing_ratio", "zero_ratio"],
    ascending=[False, False],
    na_position="last"
).reset_index(drop=True)

display(sparsity_df)

# find columns with > 90% missing
cols_to_drop = sparsity_df.loc[sparsity_df["missing_ratio"] > 0.80, "column"].tolist()

print("Columns to drop (>80% missing):")
print(cols_to_drop)
print("Number of columns to drop:", len(cols_to_drop))

# drop them
df_users = df_users.drop(columns=cols_to_drop)

print(df_users.shape)

,column,dtype,missing_count,missing_ratio,zero_count,zero_ratio,empty_string_count,empty_string_ratio
0,withheld,float64,20000,1.00000,0.0,0.00000,NaN,NaN
1,entities,float64,20000,1.00000,0.0,0.00000,NaN,NaN
2,withheld.country_codes,object,19998,0.99990,NaN,NaN,0.0,0.0
3,entities.description.cashtags,object,19851,0.99255,NaN,NaN,0.0,0.0
4,entities.description.urls,object,17471,0.87355,NaN,NaN,0.0,0.0
5,entities.description.hashtags,object,16073,0.80365,NaN,NaN,0.0,0.0
6,entities.description.mentions,object,14106,0.70530,NaN,NaN,0.0,0.0
7,pinned_tweet_id,float64,10200,0.51000,0.0,0.00000,NaN,NaN
8,entities.url.urls,object,7026,0.35130,NaN,NaN,0.0,0.0
9,location,str,5382,0.26910,NaN,NaN,NaN,NaN


Columns to drop (>80% missing):
['withheld', 'entities', 'withheld.country_codes', 'entities.description.cashtags', 'entities.description.urls', 'entities.description.hashtags']
Number of columns to drop: 6
(20000, 20)


# Inspect other relevant columns with high missing content ratio to see if it can be dropped

- mentions covers users with accounts mentioned in their description, however due to the overall sparsity of the column it should still be dropped
- urls column does not provide much meaningful data and can be dropped
- pinned_tweet_id can be more useful as a binary feature

In [5]:
cols_to_check = [
    "entities.description.mentions",
    "entities.url.urls",
]

for col in cols_to_check:
    if col in df_users.columns:
        print(f"\n=== {col} ===")
        print("dtype:", df_users[col].dtype)
        print("missing ratio:", df_users[col].isna().mean())
        print("non-missing count:", df_users[col].notna().sum())

        # look at a few non-missing examples
        display(df_users.loc[df_users[col].notna(), [col]].head(10))
    else:
        print(f"\n{col} not found in df_users.columns")

df_users["has_pinned_tweet"] = df_users["pinned_tweet_id"].notna().astype(int)

df_users = df_users.drop(
    columns=[
        c for c in [
            "entities.description.mentions",
            "entities.url.urls",
            "pinned_tweet_ids",
        ] if c in df_users.columns
    ]
)



=== entities.description.mentions ===
dtype: object
missing ratio: 0.7053
non-missing count: 5894


,entities.description.mentions
8,"[{'end': 21, 'start': 13, 'username': 'Walmart'}]"
11,"[{'end': 20, 'start': 7, 'username': 'GoogleAf..."
12,"[{'end': 11, 'start': 4, 'username': 'MPI_IS'}..."
26,"[{'end': 18, 'start': 7, 'username': 'cornellc..."
28,"[{'end': 20, 'start': 11, 'username': 'getauro..."
29,"[{'end': 20, 'start': 8, 'username': 'combrain..."
32,"[{'end': 114, 'start': 105, 'username': 'Novar..."
33,"[{'end': 102, 'start': 86, 'username': 'Unconf..."
38,"[{'end': 48, 'start': 40, 'username': 'umdclip..."
40,"[{'end': 139, 'start': 125, 'username': 'Satos..."



=== entities.url.urls ===
dtype: object
missing ratio: 0.3513
non-missing count: 12974


,entities.url.urls
0,"[{'display_url': 'boazbarak.org', 'end': 23, '..."
1,"[{'display_url': 'youtube.com/c/grian', 'end':..."
5,"[{'display_url': 'instagram.com/amnimanimals',..."
6,"[{'display_url': 'pcb.com.pk/player/fakhar-…',..."
7,"[{'display_url': 'kristina-jeromin.de', 'end':..."
9,"[{'display_url': 'vintagecomputers.code.blog',..."
10,"[{'display_url': 'dombecklab.org', 'end': 23, ..."
11,"[{'display_url': 'about.me/aniediudo', 'end': ..."
12,"[{'display_url': 'zhijing-jin.com', 'end': 23,..."
13,"[{'display_url': 'meaganmartinclimbing.com', '..."


# Fill empty text columns

In [6]:
# fill text columns
text_cols = [
    "description",
    "location",
    "name",
    "username",
    "url",
    "profile_image_url",
]

for col in text_cols:
    if col in df_users.columns:
        df_users[col] = df_users[col].fillna("unknown").astype(str).str.strip()
        df_users[col] = df_users[col].replace("", "unknown")

# Fill Empty Numeric columns

In [7]:
# convert numeric columns
numeric_cols = [
    "participation_degree",
    "public_metrics.followers_count",
    "public_metrics.following_count",
    "public_metrics.tweet_count",
    "public_metrics.listed_count",
]

for col in numeric_cols:
    if col in df_users.columns:
        df_users[col] = pd.to_numeric(df_users[col], errors="coerce").fillna(0)

# Convert created_at to datetime

In [8]:
df_users["created_at"] = pd.to_datetime(df_users["created_at"], errors="coerce", utc=True)

# Convert booleans to int

In [9]:
# convert booleans to int
bool_cols = ["protected", "verified"]

for col in bool_cols:
    df_users[col] = df_users[col].astype("boolean").fillna(False).astype(int)

# Account Age Features

In [10]:
ref_time = df_users["created_at"].max()
df_users["account_age_days"] = (ref_time - df_users["created_at"]).dt.days
df_users["account_age_days"] = df_users["account_age_days"].fillna(0).clip(lower=0)

# Rename metrics into easier column names

In [11]:
rename_map = {
    "public_metrics.followers_count": "followers_count",
    "public_metrics.following_count": "following_count",
    "public_metrics.tweet_count": "tweet_count",
    "public_metrics.listed_count": "listed_count",
}

existing_rename = {k: v for k, v in rename_map.items() if k in df_users.columns}
df_users = df_users.rename(columns=existing_rename)

df_users.head()

,id,label,split,participation_degree,created_at,description,location,name,pinned_tweet_id,profile_image_url,protected,url,username,verified,followers_count,following_count,tweet_count,listed_count,has_pinned_tweet,account_age_days
0,u1217628182611927040,human,test,1893,2020-01-16 02:02:55+00:00,Theoretical Computer Scientist. See also https...,"Cambridge, MA",Boaz Barak,NaN,https://pbs.twimg.com/profile_images/125226236...,0,https://t.co/BoMip9FF17,boazbaraktcs,0,7316,215,3098,69,0,765
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,unknown,🇬🇧,Grian,1.143808e+18,https://pbs.twimg.com/profile_images/100773461...,0,https://t.co/V3FyRYAsvK,GrianMC,0,238254,293,1400,448,1,3107
2,u1341789703633178624,bot,test,159,2020-12-23 16:56:30+00:00,unknown,unknown,mo,NaN,https://abs.twimg.com/sticky/default_profile_i...,0,unknown,mo39826506,0,0,136,6,0,0,422
3,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",unknown,AK,NaN,https://pbs.twimg.com/profile_images/145119163...,0,unknown,ak92501,0,45541,1206,9194,605,0,2855
4,u1099603759259238400,human,test,326,2019-02-24 09:35:40+00:00,unknown,unknown,Ryuu09,NaN,https://pbs.twimg.com/profile_images/121992857...,0,unknown,muffet0,0,8,289,0,0,0,1091


# Profile completeness features

In [12]:

df_users["has_description"] = (df_users["description"].str.lower() != "unknown").astype(int)

df_users["has_location"] = (df_users["location"].str.lower() != "unknown").astype(int)

df_users["has_url"] = (df_users["url"].str.lower() != "unknown").astype(int)

df_users["has_profile_image"] = (df_users["profile_image_url"].str.lower() != "unknown").astype(int)

df_users["has_pinned_tweet"] = df_users["pinned_tweet_id"].notna().astype(int)

In [13]:
profile_parts = [
    c for c in [
        "has_description",
        "has_location",
        "has_url",
        "has_profile_image",
        "has_pinned_tweet",
        "verified",
    ] if c in df_users.columns
]

if profile_parts:
    df_users["profile_completeness_score"] = df_users[profile_parts].sum(axis=1)

# Ratio Features

In [14]:
# =========================
# LOG-BASED RATIO / RATE FEATURES
# =========================

# follower / following balance
df_users["log_followers_following_ratio"] = (
    np.log1p(df_users["followers_count"]) - np.log1p(df_users["following_count"])
)

df_users["log_following_followers_ratio"] = (
    np.log1p(df_users["following_count"]) - np.log1p(df_users["followers_count"])
)

# tweet activity relative to followers / following
df_users["log_tweets_followers_ratio"] = (
    np.log1p(df_users["tweet_count"]) - np.log1p(df_users["followers_count"])
)

df_users["log_tweets_following_ratio"] = (
    np.log1p(df_users["tweet_count"]) - np.log1p(df_users["following_count"])
)

# listed relative to followers
df_users["log_listed_followers_ratio"] = (
    np.log1p(df_users["listed_count"]) - np.log1p(df_users["followers_count"])
)

# activity normalized by account age
df_users["log_normalized_followers"] = (
    np.log1p(df_users["followers_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_normalized_following"] = (
    np.log1p(df_users["following_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_normalized_tweets"] = (
    np.log1p(df_users["tweet_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

df_users["log_normalized_listed"] = (
    np.log1p(df_users["listed_count"]) - np.log1p(df_users["account_age_days"] + 1)
)

# tweet volume relative to age-adjusted behaviour
df_users["average_tweets_per_day"] = df_users["tweet_count"] / (df_users["account_age_days"] + 1)

df_users["log_tweets_per_day_relative_to_age"] = (
    np.log1p(df_users["average_tweets_per_day"]) - np.log1p(df_users["account_age_days"] + 1)
)

# broader engagement-style feature
df_users["log_tweet_engagement_ratio"] = (
    np.log1p(df_users["tweet_count"]) -
    np.log1p(df_users["followers_count"] + df_users["following_count"] + 1)
)

# Composite Indicators

In [15]:
# composite indicators
high_activity_threshold = df_users["average_tweets_per_day"].quantile(0.90)

df_users["high_activity_low_followers"] = (
    (df_users["average_tweets_per_day"] > high_activity_threshold) &
    (df_users["followers_count"] < 100)
).astype(int)

df_users["high_tweet_low_age"] = (
    (df_users["average_tweets_per_day"] > high_activity_threshold) &
    (df_users["account_age_days"] < 365)
).astype(int)

followers_threshold = df_users["followers_count"].quantile(0.10)

df_users["is_low_follower_high_activity"] = (
    (df_users["followers_count"] < followers_threshold) &
    (df_users["average_tweets_per_day"] > high_activity_threshold)
).astype(int)

tweet_threshold = df_users["tweet_count"].quantile(0.90)

df_users["new_account_high_tweets"] = (
    (df_users["account_age_days"] < 365) &
    (df_users["tweet_count"] > tweet_threshold)
).astype(int)

df_users["follows_more_than_followed"] = (
    df_users["following_count"] > df_users["followers_count"]
).astype(int)

df_users["is_new_with_verified"] = (
    (df_users["account_age_days"] < 365) & (df_users["verified"] == 1)
).astype(int)


# account age category
df_users["account_age_category"] = pd.cut(
    df_users["account_age_days"],
    bins=[0, 365, 730, 1825, float("inf")],
    labels=["Very New", "New", "Established", "Old"]
).cat.codes

# Text Based Features

In [ ]:
# =========================
# TEXT FEATURES
# =========================

df_users["description_length"] = df_users["description"].apply(lambda x: len(x))

df_users["username_length"] = df_users["username"].apply(lambda x: len(x))

df_users["description_word_count"] = df_users["description"].apply(lambda x: len(x.split()))

df_users["has_url_in_description"] = df_users["description"].apply(
    lambda x: int("http" in x.lower() or "www" in x.lower())
)

df_users["description_special_chars_count"] = df_users["description"].apply(
    lambda x: sum(1 for c in x if c in "!@#$%^&*") if isinstance(x, str) else 0
)

df_users["description_uppercase_ratio"] = df_users["description"].apply(
    lambda x: sum(c.isupper() for c in x) / len(x) if len(x) > 0 else 0
)

df_users["username_special_chars_ratio"] = df_users["username"].apply(
    lambda x: sum(1 for c in x if not c.isalnum()) / len(x) if len(x) > 0 else 0
)

df_users["username_digit_count"] = df_users["username"].str.count(r"\d")

df_users["username_upper_ratio"] = df_users["username"].apply(
    lambda x: sum(c.isupper() for c in x) / len(x) if len(x) > 0 else 0
)

df_users["username_underscore_count"] = df_users["username"].str.count("_")

# Other Feature columns

In [17]:
# disparity feature
df_users["follower_following_disparity"] = abs(
    df_users["followers_count"] - df_users["following_count"]
)

# disparity feature
df_users["follower_following_disparity"] = abs(
    df_users["followers_count"] - df_users["following_count"]
)

# spam-word score
spam_words = [
    'win', 'free', 'offer', 'click', 'buy', 'subscribe', 
    'act now', 'apply now', 'call now', 'don’t hesitate', 
    'for only', 'get started now', 'limited time', 'great offer', 
    'instant', 'now only', 'offer expires', 'once in a lifetime', 
    'order now', 'order today', 'special promotion', 'urgent', 
    'while supplies last', 'bonus', 'all new', 'amazing', 
    'certified', 'congratulations', 'fantastic deal', 'for free', 
    'guaranteed', 'outstanding value', 'risk free', 
    'satisfaction guaranteed', 'free!', 'free trial', 'free consultation', 
    'free gift', 'free membership', 'free offer', 'free preview', 
    'free sample', 'free quote', 'sign up free today', 'deal', 
    'giving away', 'no obligation', 'no strings attached', 'offer', 
    'prize', 'trial', 'unlimited', 'what are you waiting for?', 
    'win', 'winner', 'you’re a winner!', 'won', 'you have been selected', 
    '#1', '100% free', '100% satisfied', '50% off', 
    'one hundred percent guaranteed', 'click below', 'click here', 
    'increase sales', 'increase your sales', 'opt in', 'open', 'sale', 
    'sales', 'subscribe', 'chance', 'sample', 'satisfaction', 'solution', 
    'success', 'cards accepted', 'full refund', 'affordable', 
    'bargain', 'best price', 'cash', 'cash bonus', 'cheap', 
    'credit', 'discount', 'for just $', 'lowest price', 'save big money', 
    'why pay more?', 'buy', 'as seen on', 'buy direct', 'clearance', 
    'order', '$$$', 'marketing solutions', 'join millions', 
    'name brand', 'no questions asked', 'giving it away', 
    'best rates', 'compare', 'drastically reduced'
]

df_users["description_spam_score"] = df_users["description"].apply(
    lambda x: sum(word in x.lower() for word in spam_words) if isinstance(x, str) else 0
)

In [18]:
df_users.head()


,id,label,split,participation_degree,created_at,description,location,name,pinned_tweet_id,profile_image_url,...,username_length,description_word_count,has_url_in_description,username_digit_ratio,description_digit_ratio,description_special_chars_count,description_uppercase_ratio,username_special_chars_ratio,follower_following_disparity,description_spam_score
0,u1217628182611927040,human,test,1893,2020-01-16 02:02:55+00:00,Theoretical Computer Scientist. See also https...,"Cambridge, MA",Boaz Barak,NaN,https://pbs.twimg.com/profile_images/125226236...,...,12,8,1,0.000000,0.021739,0,0.184783,0.0,7101,0
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,unknown,🇬🇧,Grian,1.143808e+18,https://pbs.twimg.com/profile_images/100773461...,...,7,1,0,0.000000,0.000000,0,0.000000,0.0,237961,0
2,u1341789703633178624,bot,test,159,2020-12-23 16:56:30+00:00,unknown,unknown,mo,NaN,https://abs.twimg.com/sticky/default_profile_i...,...,10,1,0,0.800000,0.000000,0,0.000000,0.0,136,0
3,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",unknown,AK,NaN,https://pbs.twimg.com/profile_images/145119163...,...,7,5,0,0.714286,0.000000,0,0.000000,0.0,44335,1
4,u1099603759259238400,human,test,326,2019-02-24 09:35:40+00:00,unknown,unknown,Ryuu09,NaN,https://pbs.twimg.com/profile_images/121992857...,...,7,1,0,0.142857,0.000000,0,0.000000,0.0,281,0


# TF-IDF on Description Column

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

# make sure description is clean text
df_users["description"] = df_users["description"].fillna("unknown").astype(str)

tfidf = TfidfVectorizer(
    max_features=300,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8
)

description_tfidf = tfidf.fit_transform(df_users["description"])

tfidf_feature_names = [f"desc_tfidf_{f}" for f in tfidf.get_feature_names_out()]

df_description_tfidf = pd.DataFrame(
    description_tfidf.toarray(),
    columns=tfidf_feature_names,
    index=df_users.index
)

df_users = pd.concat([df_users, df_description_tfidf], axis=1)
print(df_users.shape)

(20000, 366)


# Drop Raw Text Columns

In [22]:
cols_to_drop = [
    "created_at",
    "username",
    "name",
    "description",
    "location",
    "url",
    "profile_image_url",
    "entities.description.urls",
    "entities.description.mentions",
    "entities.description.hashtags",
    "entities.description.cashtags",
]

df_users_model = df_users.drop(columns=[c for c in cols_to_drop if c in df_users.columns])

# Merge user metadata FE with tweets FE (file from FE_tweets.ipynb)

In [ ]:
import pandas as pd

df_users_model

df_tweets_model = pd.read_parquet("./datasets/final_outputs/df_tweets_model.parquet")

df_merged = df_users_model.merge(
    df_tweets_model,
    on="id",
    how="left"
)

print(df_merged.shape)

df_merged.to_parquet("./datasets/final_outputs/df_user_meta_full.parquet", index=False)


(20000, 399)
